In [1]:
import sys
sys.path.append('..')
from osp import *

In [2]:
df_meta = get_corpus_metadata()
df_meta.iloc[0]

uuid                      f6eecd30-8c4a-3c3d-a9da-4e777d091b2c
title                                "Aristotlés" Horror Vacui
author                                              John Thorp
year                                                      1990
journal                         Canadian Journal of Philosophy
volume                                                      20
issue                                                        2
url                          jstor.org/stable/10.2307/40231690
publisher                           Cambridge University Press
discipline                                          Philosophy
discipline_names                                           NaN
decade                                                    1990
period                                               1975-2000
century                                                    C20
halfcentury                                               lC20
century_discipline                              C20 Phi

In [3]:
df_meta.discipline.value_counts()

discipline
Philosophy    32277
Other         32277
Literature    25343
Name: count, dtype: int64

In [4]:
parsed_ids = set(get_parsed_slice_ids())
# len(parsed_ids)

In [5]:
parsed_counts = Counter()
for x in parsed_ids:
    parsed_counts[x.split("__")[0]] += 1
# parsed_counts

In [6]:
slice_counts = {x:len(y) for x,y in tqdm(STASH_SLICES.items(), total=len(STASH_SLICES))}

100%|██████████| 89897/89897 [00:14<00:00, 6264.83it/s]


In [7]:
df_meta['num_articles'] = 1
df_meta['num_slices'] = [slice_counts[x] for x in df_meta.index]
df_meta['num_parsed_slices'] = [parsed_counts[x] for x in df_meta.index]
df_meta[['num_slices', 'num_parsed_slices']]


,num_slices,num_parsed_slices
id,,
phil/10.2307/40231690,4,1
phil/10.2307/40230399,1,0
phil/10.2307/40231533,1,1
phil/10.2307/40230622,1,1
phil/10.2307/40231225,0,0
...,...,...
other/10.2307/44827520,0,0
other/10.2307/44823154,0,0
other/10.2307/44823659,0,0


In [8]:
aggs = []
for disc in ['Philosophy', 'Literature', 'Other']:

    agg = df_meta.query('discipline == @disc').groupby(['discipline','period']).agg(
        {
            'journal':'nunique',
            'author':'nunique',
            'num_articles': 'sum',
            'num_slices': 'sum',
            'num_parsed_slices': 'sum',
        }
    )
    aggs.append(agg)

odf = pd.concat(aggs)
odf.columns = ['# Journals', '# Authors', '# Articles', '# Slices', '# Parsed Slices']
odf = odf.applymap(lambda x: f'{int(x):,d}')
odf

# Journals # Authors # Articles # Slices # Parsed Slices
discipline period                                                            
Philosophy 1900-1925          4       841      1,828    7,498           5,011
           1925-1950          9     1,739      4,220   12,096           5,013
           1950-1975         10     3,288      6,760   19,095           5,023
           1975-2000         11     4,776      8,369   31,959           5,737
           2000-2025         11     7,462     11,100   53,749           9,654
Literature 1900-1925          3     1,064      2,458    4,238           4,238
           1925-1950          5     2,639      5,080    7,357           5,013
           1950-1975          7     3,791      5,510   12,224           5,011
           1975-2000          7     4,657      6,378   21,199           5,028
           2000-2025          7     4,658      5,917   17,293           5,018
Other      1900-1925        312     1,211      1,828    1,587           1,587
           1925-1950        645     3,315      4,220    3,966           3,966
           1950-1975      1,142     5,892      6,760    6,679           5,009
           1975-2000      1,771     7,611      8,369   10,373           5,016
           2000-2025      1,971     9,783     11,100   15,402           5,013

In [9]:
print(df_to_latex_table(odf.reset_index()))

\begin{table}[H]
  \centering
  \small
  \begin{tabular}{lllllll}
  \toprule
  discipline & period & \# Journals & \# Authors & \# Articles & \# Slices & \# Parsed Slices \\
  \midrule
  Philosophy & 1900-1925 & 4 & 841 & 1,828 & 7,498 & 5,011 \\
  Philosophy & 1925-1950 & 9 & 1,739 & 4,220 & 12,096 & 5,013 \\
  Philosophy & 1950-1975 & 10 & 3,288 & 6,760 & 19,095 & 5,023 \\
  Philosophy & 1975-2000 & 11 & 4,776 & 8,369 & 31,959 & 5,737 \\
  Philosophy & 2000-2025 & 11 & 7,462 & 11,100 & 53,749 & 9,654 \\
  Literature & 1900-1925 & 3 & 1,064 & 2,458 & 4,238 & 4,238 \\
  Literature & 1925-1950 & 5 & 2,639 & 5,080 & 7,357 & 5,013 \\
  Literature & 1950-1975 & 7 & 3,791 & 5,510 & 12,224 & 5,011 \\
  Literature & 1975-2000 & 7 & 4,657 & 6,378 & 21,199 & 5,028 \\
  Literature & 2000-2025 & 7 & 4,658 & 5,917 & 17,293 & 5,018 \\
  Other & 1900-1925 & 312 & 1,211 & 1,828 & 1,587 & 1,587 \\
  Other & 1925-1950 & 645 & 3,315 & 4,220 & 3,966 & 3,966 \\
  Other & 1950-1975 & 1,142 & 5,892 & 6,760 

In [10]:
s=df_meta[df_meta['discipline'] == 'Literature'].journal.value_counts()
l = []
for k,v in s.items():
    ymin = df_meta[df_meta.journal == k].year.min()
    ymax = df_meta[df_meta.journal == k].year.max()
    l.append(f'{k} ({ymin}-{ymax}, n={v:,})')
print('\n'.join(l))

PMLA (1900-2016, n=7,258)
The Modern Language Review (1905-2018, n=5,698)
The Review of English Studies (1925-2016, n=3,008)
Modern Philology (1903-2018, n=2,997)
ELH (1934-2016, n=2,619)
New Literary History (1969-2016, n=1,928)
Critical Inquiry (1974-2019, n=1,835)


In [11]:
len(l)

7

In [12]:
def get_num_parsed_slices(id):
    return len([x for x in parsed_ids if x.startswith(id)])

df_meta['num_parsed_slices'] = [get_num_parsed_slices(id) for id in tqdm(df_meta.index)]

  8%|▊         | 7240/89897 [00:36<07:12, 191.27it/s]

KeyboardInterrupt: 

In [14]:
df_feats = pd.read_pickle('../data/raw/df_feats3.pkl.gz')
df_feats['weight_abs'] = df_feats['weight'].abs()
df_feats = df_feats.groupby('feature').mean(numeric_only=True).reset_index().drop(columns=['run']).sort_values('weight_abs',ascending=False)    
# df_feats1 = df_feats[df_feats.comparison.str.contains('Philosophy') & df_feats.comparison.str.contains('Other')].groupby('feature').mean(numeric_only=True).reset_index().drop(columns=['run']).sort_values('weight_abs',ascending=False)
# df_feats2 = df_feats1.copy()
# df_feats2['feat_type'] = df_feats2['feature'].apply(lambda x: x.split('_')[0])
# df_feats2['num'] = 1

# df_feats2.groupby('feat_type').agg(
#     {
#         'num': 'sum',
#         'weight_abs': 'max',
#     }
# ).sort_values('num',ascending=False)

In [15]:
df = STASH_SLICE_FEATS.df

In [16]:
df0 = df[[c for c in df.columns if c not in BAD_SLICE_FEATS and not c.startswith('phrase_') and not c.startswith('ttr_')]]
df0

,pos_DT,pos_VBZ,pos_RB,pos_JJ,pos_IN,pos_NN,pos_PRP$,pos_NNS,pos_TO,pos_VB,...,deprel_dislocated,deprel_csubj:pass,deprel_advcl:relcl,pos_NFP,deprel_vocative,pos_SYM,deprel_orphan,pos_AFX,deprel_goeswith,pos_GW
_key,,,,,,,,,,,,,,,,,,,,,
phil/10.2307/2380200__04,84.104289,46.257359,81.581161,116.904962,111.017662,102.607233,15.979815,45.416316,22.708158,58.031960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/2514892__02,87.415222,20.346647,16.578749,97.211756,143.180106,113.790505,5.275057,66.314996,5.275057,11.303693,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/14710__03,102.811245,11.244980,46.586345,100.401606,130.120482,167.068273,12.048193,52.208835,8.032129,17.670683,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/20111773__02,98.193244,56.559309,56.559309,98.193244,135.113904,109.190888,6.284368,35.349568,9.426551,24.351925,...,0.785546,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/2379466__05,91.644205,32.345013,65.588500,79.065588,139.263252,141.958670,11.680144,53.908356,12.578616,40.431267,...,NaN,0.898473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lit/468421__06,116.161616,40.404040,44.612795,75.757576,136.363636,162.457912,11.784512,54.713805,14.309764,37.878788,...,NaN,0.841751,0.841751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/29782027__24,80.839895,18.372703,29.921260,51.968504,70.341207,87.664042,0.524934,25.721785,7.349081,25.721785,...,NaN,NaN,NaN,5.249344,NaN,0.524934,NaN,NaN,NaN,NaN
lit/459531__04,92.471358,40.098200,36.824877,80.196399,135.842881,134.206219,18.003273,49.099836,21.276596,32.733224,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
dfz = df0.copy().fillna(0)
for c in dfz.columns:
    dfz[c] = (dfz[c] - dfz[c].mean()) / dfz[c].std()
dfz['discipline'] = dfz.index.map(lambda x: x.split('/')[0])
dfz

,pos_DT,pos_VBZ,pos_RB,pos_JJ,pos_IN,pos_NN,pos_PRP$,pos_NNS,pos_TO,pos_VB,...,deprel_csubj:pass,deprel_advcl:relcl,pos_NFP,deprel_vocative,pos_SYM,deprel_orphan,pos_AFX,deprel_goeswith,pos_GW,discipline
_key,,,,,,,,,,,,,,,,,,,,,
phil/10.2307/2380200__04,-0.971340,0.842103,3.045618,2.142348,-1.341446,-1.599537,0.547509,-0.159382,1.635407,2.212708,...,-0.353549,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,phil
other/10.2307/2514892__02,-0.793884,-0.942321,-2.223544,1.108831,0.433220,-1.207474,-0.946357,0.867249,-1.367570,-1.633699,...,-0.353549,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,other
other/10.2307/14710__03,0.031297,-1.569136,0.208902,1.276237,-0.287387,0.660334,-0.001156,0.174295,-0.892644,-1.109605,...,-0.353549,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,other
phil/10.2307/20111773__02,-0.216215,1.551580,1.017321,1.160340,-0.011859,-1.368727,-0.805506,-0.653903,-0.652445,-0.559643,...,-0.353549,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,phil
phil/10.2307/2379466__05,-0.567223,-0.116015,1.749236,0.156503,0.217095,-0.219957,-0.052517,0.257783,-0.109479,0.763918,...,2.239801,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,phil
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lit/468421__06,0.746837,0.438996,0.048924,-0.017104,0.057099,0.498704,-0.037953,0.297350,0.188723,0.553812,...,2.076079,0.502096,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,lit
other/10.2307/29782027__24,-1.146302,-1.078263,-1.141986,-1.265576,-3.585900,-2.123414,-1.609244,-1.126860,-1.010304,-0.446883,...,-0.353549,-0.640164,4.888631,-0.388949,0.269162,-0.174895,-0.145034,-0.01923,-0.0456,other
lit/459531__04,-0.522890,0.417933,-0.582372,0.215849,0.028365,-0.491742,0.829885,0.021568,1.388810,0.130259,...,-0.353549,-0.640164,-0.227423,-0.388949,-0.199913,-0.174895,-0.145034,-0.01923,-0.0456,lit


In [18]:
stash_eg = get_stash('osp_feat_examples3')
stash_eg_df = stash_eg.df
feat2eg = dict(zip(stash_eg_df.feature, stash_eg_df.eg_text))
# feat2eg

In [19]:
stash_eg2 = get_stash('osp_feat_examples2')
l = []
for id,idld in tqdm(stash_eg2.items(), total=len(stash_eg2)):
    if id.startswith('phil'):
        idd = random.choice(idld)
        l.append(idd)

100%|██████████| 33666/33666 [01:14<00:00, 449.21it/s]


In [20]:
ldf = pd.DataFrame(l).sample(frac=1)
ldf['eg_text_len'] = ldf.eg_text.apply(len)
ldf.sort_values('eg_text_len',ascending=False,inplace=True)
ldf.drop_duplicates(subset='feature',inplace=True)
ldf.feature.nunique()

98

In [21]:

def shorten_eg(eg, max_len=40):
    eg_pre,w,eg_post = eg.split('*',2)
    radius = (max_len - len(w)) // 2
    eg_pre = eg_pre[-radius:]
    eg_post = eg_post[:max_len-len(eg_pre)-len(w)]
    return f'{eg_pre}XXXEMPHXXX{{{w.lower()}}}{eg_post}'

ldf['eg_short'] = ldf.eg_text.apply(shorten_eg)

feat2eg = dict(zip(ldf.feature, ldf.eg_short))
# ldf

In [22]:
dftf = df0.copy().fillna(0)
dftf['discipline'] = dftf.index.map(lambda x: x.split('/')[0])
dftf = dftf.groupby('discipline').mean().T.round(0).applymap(int).applymap(str)
feat2tf = dict(zip(dftf.index, dftf.phil))
# feat2tf

In [30]:
df_feats = pd.read_pickle('../data/raw/df_feats3.pkl.gz')
df_feats = df_feats[df_feats.comparison.str.contains('Philosophy')]
df_feats['period1'] = df_feats.comparison.str.split().str[0]
df_feats['period2'] = df_feats.comparison.str.split().str[-2]
df_feats = df_feats[df_feats.period1 == df_feats.period2]
df_feats.query('feature=="deprel_mark"').sort_values('weight',ascending=False)
feat2weight = df_feats.groupby('feature').mean(numeric_only=True).weight
feat2weight

feature
deprel_acl            0.005139
deprel_acl:relcl     -0.011177
deprel_advcl         -0.135338
deprel_advcl:relcl    0.100349
deprel_advmod         0.368142
                        ...   
sent_DC               0.174540
sent_DCw             -0.125420
sent_IC              -0.002408
sent_ICw             -0.170539
sent_Wd               0.303272
Name: weight, Length: 94, dtype: float64

In [33]:
dfagg = dfz.groupby('discipline').mean().T.sort_values('phil',ascending=False).rename_axis('feature')
# dfagg = dfagg.round(2).applymap(lambda x: f'+{x}' if x>0 else f'{x}')
dfagg = dfagg.round(2).applymap(str)
dfagg['feat_desc'] = dfagg.index.map(lambda x: f'{FEAT2DESC[x]} ({x.split("_")[0]})')
dfagg['eg'] = dfagg.index.map(lambda x: '' if x not in feat2eg else '...'+feat2eg[x].strip()+'...')
dfagg['weight'] = dfagg.index.map(feat2weight)
dfagg['weight'] = dfagg['weight'].apply(lambda x: f'{"+" if x>0 else ""}{x:.2f}')
dfagg['phil_tf'] = dfagg.index.map(feat2tf)
# dfagg['feat_type'] = dfagg.index.map(lambda x: x.split('_')[0])
# dfagg = dfagg[['feat_desc','eg','phil','lit','other']].fillna('').reset_index()
dfagg = dfagg[['feat_desc','eg','phil_tf','phil','weight']].fillna('').reset_index().drop(columns=['feature'])
dfagg.columns = ['Feature', 'Example', 'Average', 'Z-score','Weight']
# dfagg = dfagg.set_index(['Feature','Description','Example'])
dfagg

,Feature,Example,Average,Z-score,Weight
0,Copula (deprel),...terpart relations XXXEMPHXXX{are} operative in (ii)...,27,0.65,+0.64
1,Marker (deprel),...particularly care XXXEMPHXXX{if} it is understood o...,45,0.62,+0.63
2,# Dependent clauses (sent),,79,0.54,+0.17
3,Modal (pos),...n mathematics can XXXEMPHXXX{can} interpreted to com...,16,0.54,+0.34
4,"Verb, 3rd person sing. pres. (pos)",...ble disease that XXXEMPHXXX{pushes} the reduced popu...,42,0.53,+0.32
...,...,...,...,...,...
89,Compound (deprel),...that the various XXXEMPHXXX{art} objects and activi...,14,-0.33,-0.66
90,"Proper noun, plural (pos)",...ills allow Homo XXXEMPHXXX{sapiens} an 'internal acc...,1,-0.39,-0.29
91,# Words in independent clauses (sent),,376,-0.46,-0.17
92,"Verb, past tense (pos)","...human purpose, XXXEMPHXXX{expressed} by saying that...",6,-0.48,-0.39


In [34]:
out=df_to_latex_table(dfagg.head(25), caption="""
Top 25 most distinctive features of philosophy articles. Averages reflect average frequency per 1,000 words. Z-scores express the number of standard deviations from the mean frequency across all articles in the corpus.
""".strip())
out = out.replace("XXXEMPHXXX\{","\\textbf{")
out = out.replace("\\}","}")
print(out)

\begin{table}[H]
  \centering
  \small
  \begin{tabular}{lllll}
  \toprule
  Feature & Example & Average & Z-score & Weight \\
  \midrule
  Copula (deprel) & ...terpart relations \textbf{are} operative in (ii)... & 27 & 0.65 & +0.64 \\
  Marker (deprel) & ...particularly care \textbf{if} it is understood o... & 45 & 0.62 & +0.63 \\
  \# Dependent clauses (sent) &  & 79 & 0.54 & +0.17 \\
  Modal (pos) & ...n mathematics can \textbf{can} interpreted to com... & 16 & 0.54 & +0.34 \\
  Verb, 3rd person sing. pres. (pos) & ...ble disease that \textbf{pushes} the reduced popu... & 42 & 0.53 & +0.32 \\
  Verb, base form (pos) & ...mizing, i.e., must \textbf{be} the act whose perf... & 37 & 0.52 & +0.24 \\
  \# Clause transitions (sent) &  & 126 & 0.51 & +0.16 \\
  Clausal complement (deprel) & ...genous components \textbf{map} onto valence, a pr... & 9 & 0.5 & +0.34 \\
  Expletive (deprel) & ...a point, so that \textbf{there} will be no distan... & 6 & 0.47 & +0.04 \\
  \# Unique clauses (sen

In [39]:
with open('../../../Dropbox/Prof/Articles/OSP/tables/table.topfeats.tex', 'w') as f:
    f.write(out)
